# Chapitre 0.5 : un premier réseau

Un seul neurone sait imiter les portes logiques ET et OU, mais pas le OU exclusif. Dans ce carnet, tu le vérifies, puis tu construis un petit réseau de 3 neurones qui, lui, l’apprend. Tu l’entraînes d’abord à la main, en Python pur, avec l’erreur et la descente de gradient de 0.2. Puis tu refais le même réseau avec PyTorch, et tu vérifies qu’il trouve les mêmes nombres.

Ce carnet accompagne [la page du chapitre 0.5](https://chipat-neko.github.io/forge-ia/partie-0/0-5/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. régler un neurone à seuil pour les portes ET, OU et OU exclusif ;
2. construire un réseau 2-2-1, et y faire passer les 4 points ;
3. calculer ses dérivées à la main, de la sortie vers l’entrée : la rétropropagation ;
4. l’entraîner, et suivre la courbe de sa perte ;
5. refaire le même réseau dans PyTorch, et comparer les dérivées ;
6. choisir l’appareil, le processeur ou la carte graphique ;
7. relever un défi, puis vérifier que ce carnet trouve les mêmes nombres que la page du site.

**Comment l’utiliser**

Il n’y a rien à installer : Colab a déjà Python et PyTorch, et ce carnet ne les installe pas. Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Change les nombres des cellules et relance-les : c’est fait pour.

**Pour avoir une carte graphique dans Colab** : menu « Exécution », puis « Modifier le type d’exécution ». Choisis un accélérateur « GPU », puis enregistre. Colab redémarre : relance les cellules depuis le début. L’offre gratuite ne garantit pas de carte, et une session dure au plus 12 heures. Sans carte, tout marche aussi, sur le processeur.

## Étape 0 : récupérer le code

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, puis indique à Python où trouver le projet. Le projet utilise PyTorch, que Colab a déjà installé.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia

projet = depot / "projets" / "0.5-reseau"
if not projet.exists():
    raise FileNotFoundError("Le dépôt n’a pas pu être téléchargé. Vérifie ta connexion, puis relance cette cellule.")

# Python cherche les modules dans les dossiers de sys.path : on y ajoute le projet.
sys.path.insert(0, str(projet.resolve()))
print("Le code du projet est prêt.")

## Étape 1 : un neurone à seuil

Un neurone reçoit ici deux entrées, x1 et x2, qui valent 0 ou 1. Il en fait une **somme pondérée** : w1 × x1 + w2 × x2 + b. Les **poids** w1 et w2 disent l’importance de chaque entrée, et le **biais** b décale le tout. Le neurone à seuil répond 1 si la somme est plus grande que 0, et 0 sinon.

Une **porte logique** donne une réponse pour chacun des 4 points (0, 0), (0, 1), (1, 0) et (1, 1) :

- ET répond 1 seulement si les deux entrées valent 1 ;
- OU répond 1 si au moins une entrée vaut 1 ;
- le OU exclusif répond 1 si une seule des deux entrées vaut 1.

Avec w1 = w2 = 1 et b = −1,5, la somme ne dépasse 0 que pour (1, 1) : c’est la porte ET. `sortie_seuil` donne la réponse du neurone, et `bien_classes` compte les points où elle est juste.

In [ ]:
from premier_reseau.neurone import GRILLE, POINTS, PORTES, bien_classes, meilleur_reglage, sortie_seuil

# Le neurone de la porte ET : w1 = w2 = 1, et b = -1,5.
for x in POINTS:
    print(f"{x}\u202f: le neurone répond {sortie_seuil((1.0, 1.0), -1.5, x)}")
print(f"Points bien classés pour la porte ET\u202f: {bien_classes((1.0, 1.0), -1.5, 'et')} sur 4")

`meilleur_reglage` essaie tous les réglages d’une grille : w1, w2 et b de −2 à 2, par pas de 0,5, soit 9 × 9 × 9 = 729 réglages. Pour chaque porte, elle garde le premier réglage qui classe le plus de points.

In [ ]:
print(f"{len(GRILLE) ** 3} réglages essayés, avec w1, w2 et b dans {GRILLE}")
for porte in PORTES:
    compte, poids, biais = meilleur_reglage(porte)
    print(f"{porte}\u202f: {compte} points sur 4 au mieux, avec les poids {poids} et le biais {biais}")

Pour le OU exclusif, c’est 3 points sur 4 au mieux, et pas seulement sur cette grille. Le neurone coupe le plan des points en deux par une ligne droite, celle où la somme pondérée vaut 0 : il répond 1 d’un côté, et 0 de l’autre. Or aucune ligne droite ne met (0, 1) et (1, 0) d’un côté, et (0, 0) et (1, 1) de l’autre. Il faut plusieurs neurones.

## Étape 2 : un réseau 2-2-1

Le réseau a 2 entrées, une **couche cachée** de 2 neurones, et 1 neurone de sortie : c’est un réseau 2-2-1. Chaque neurone caché fait sa somme pondérée des deux entrées. Le neurone de sortie fait la somme pondérée des deux neurones cachés.

Le seuil, qui saute de 0 à 1, est remplacé par la **sigmoïde**, 1 / (1 + exp(−z)), qui monte en douceur de 0 à 1. Grâce à elle, le réseau a une pente partout, et la descente de gradient de 0.2 peut s’en servir. La sortie p est un nombre entre 0 et 1 : la probabilité, selon le réseau, que la réponse soit 1.

Le réseau a 9 poids : 4 entre les entrées et la couche cachée, 2 biais cachés, 2 poids vers la sortie et 1 biais de sortie. `initialiser(GRAINE)` les tire au hasard entre −1 et 1. `propager` fait traverser le réseau à un point, de l’entrée vers la sortie.

In [ ]:
from premier_reseau.a_la_main import DONNEES, GRAINE, initialiser, nb_poids, propager, sigmoide

print(f"sigmoide(-4) = {sigmoide(-4):.3f}, sigmoide(0) = {sigmoide(0)}, sigmoide(4) = {sigmoide(4):.3f}")
reseau = initialiser(GRAINE)
print(f"{nb_poids(reseau)} poids, tirés au hasard avec la graine {GRAINE}\u202f:")
print(reseau)
for x, y in DONNEES:
    print(f"{x}\u202f: le réseau donne {propager(reseau, x).p:.3f}, il faudrait {y}")

## Étape 3 : l’erreur et la rétropropagation

Pour chaque point, l’erreur est celle de 0.2 : −ln de la probabilité donnée à la bonne réponse, soit −ln p si la réponse est 1, et −ln(1 − p) si c’est 0. La **perte** est la moyenne des erreurs sur les 4 points : c’est elle que l’entraînement fait baisser.

Pour savoir dans quel sens bouger chaque poids, il faut la dérivée de la perte par rapport à lui. `retropropager` les calcule toutes, de la sortie vers l’entrée, avec la règle de la chaîne de 0.2 :

- à la sortie, la sigmoïde et l’erreur se dérivent ensemble en p − y, où y est la bonne réponse ;
- un poids vers la sortie reçoit p − y, fois la sortie du neurone caché qu’il relie ;
- chaque neurone caché reçoit sa part de p − y, selon son poids vers la sortie, fois la pente de sa sigmoïde, h × (1 − h) ;
- enfin, un poids d’entrée reçoit la part de son neurone, fois l’entrée qu’il relie.

`gradient_moyen` en fait la moyenne sur les 4 points. La cellule vérifie l’une des 9 dérivées, celle du biais de sortie b2, par la pente approchée de 0.2 : on décale b2 d’un tout petit h, de chaque côté, et on regarde de combien la perte change.

In [ ]:
from premier_reseau.a_la_main import Reseau, gradient_moyen, perte_moyenne
from premier_reseau.nombres import en_francais, en_puissance_de_dix

g = gradient_moyen(reseau, DONNEES)
print(f"Perte au départ\u202f: {en_francais(perte_moyenne(reseau, DONNEES), 3)}")
print(g)

# La dérivée par rapport à b2, par la pente approchée : (perte(b2 + h) - perte(b2 - h)) / 2h.
h = 1e-6
plus = Reseau(reseau.w1, reseau.b1, reseau.w2, reseau.b2 + h)
moins = Reseau(reseau.w1, reseau.b1, reseau.w2, reseau.b2 - h)
approchee = (perte_moyenne(plus, DONNEES) - perte_moyenne(moins, DONNEES)) / (2 * h)
print(f"Dérivée par rapport à b2\u202f: {g.b2:.6f} par la rétropropagation, {approchee:.6f} par la pente approchée")
print(f"Écart entre les deux\u202f: {en_puissance_de_dix(abs(g.b2 - approchee))}")

## Étape 4 : l’entraînement, et la courbe de sa perte

Un pas de descente recule chaque poids de taux × sa dérivée, comme en 0.2. `entrainer` fait 3 000 pas, avec un taux de 1, sur les 4 points à la fois. Il note la perte avant le premier pas, puis après chaque pas.

La cellule dessine la courbe en texte : une ligne tous les 250 pas, avec une barre d’autant plus longue que la perte est grande. Puis elle donne la réponse du réseau entraîné pour chaque point.

In [ ]:
from premier_reseau.a_la_main import NB_PAS, TAUX, entrainer, predictions

historique = entrainer(reseau, DONNEES, TAUX, NB_PAS)
for numero in range(0, NB_PAS + 1, 250):
    perte = historique.pertes[numero]
    print(f"pas {en_francais(numero):>5}\u202f: {'█' * round(perte * 50):<45}{en_francais(perte, 3)}")
print()
for (x, y), p in zip(DONNEES, predictions(historique.reseau, DONNEES)):
    print(f"{x}\u202f: {p:.3f}, arrondi à {round(p)}, il faudrait {y}")

Le réseau n’apprend pas à tous les coups. Avec d’autres poids de départ, il peut rester coincé : la perte cesse de baisser avant que les 4 réponses soient justes. C’est ce qui arrive avec la graine 1234, la première prévue pour ce chapitre. La cellule montre sa perte finale et ses réponses.

In [ ]:
coince = entrainer(initialiser(1234), DONNEES, TAUX, NB_PAS)
print(f"Graine 1234\u202f: perte finale {en_francais(coince.pertes[-1], 3)}")
for (x, y), p in zip(DONNEES, predictions(coince.reseau, DONNEES)):
    print(f"{x}\u202f: {p:.3f}, arrondi à {round(p)}, il faudrait {y}")

## Étape 5 : le même réseau dans PyTorch

PyTorch range les mêmes calculs dans des couches : `nn.Linear(2, 2)` fait les deux sommes pondérées de la couche cachée, et `nn.Sigmoid()` applique la sigmoïde. `construire(reseau)` empile les quatre couches, en `float64`, et y recopie les 9 poids de départ.

Avec PyTorch, plus besoin de dériver à la main : `perte.backward()` remonte le calcul et range chaque dérivée à côté de son poids. C’est **autograd**, la dérivation automatique. `gradients_pytorch` la lance avec `nn.BCELoss()`, l’erreur de l’étape 3, en moyenne sur les 4 points. Les deux calculs doivent donner les mêmes dérivées, aux arrondis du dernier chiffre près.

Puis `entrainer_pytorch` refait les 3 000 pas, avec l’optimiseur `torch.optim.SGD`, qui recule chaque poids de taux × sa dérivée.

In [ ]:
import torch

from premier_reseau.avec_pytorch import construire, entrainer_pytorch, gradients_pytorch, plus_grand_ecart

modele = construire(reseau)
print(modele)
ecart = plus_grand_ecart(gradient_moyen(reseau, DONNEES), gradients_pytorch(modele, DONNEES))
print(f"PyTorch {torch.__version__}, plus grand écart avec les dérivées à la main\u202f: {en_puissance_de_dix(ecart)}")

sur_le_processeur = entrainer_pytorch(construire(reseau), DONNEES, TAUX, NB_PAS, "cpu")
print(
    f"{en_francais(NB_PAS)} pas sur le processeur en {en_francais(sur_le_processeur.millisecondes)} ms, "
    f"perte finale {en_francais(sur_le_processeur.pertes[-1], 3)}"
)
ecart_final = plus_grand_ecart(historique.reseau, sur_le_processeur.reseau)
print(f"Plus grand écart des poids finaux avec la main\u202f: {en_puissance_de_dix(ecart_final)}")

## Étape 6 : l’appareil

`choisir_appareil()` dit où calculer, comme dans les projets 0.3 et 0.4 : la variable FORGE_APPAREIL si elle est réglée, sinon la carte graphique si PyTorch en voit une, sinon le processeur. Si c’est la carte, la cellule refait l’entraînement sur elle, et compare ses poids finaux à ceux du processeur. Les durées dépendent de la machine : compare-les toi-même.

In [ ]:
from premier_reseau.appareil import choisir_appareil

ou = choisir_appareil()
if ou.type == "cuda":
    sur_la_carte = entrainer_pytorch(construire(reseau), DONNEES, TAUX, NB_PAS, "cuda")
    print(
        f"{en_francais(NB_PAS)} pas sur la carte graphique, {torch.cuda.get_device_name(0)}, "
        f"en {en_francais(sur_la_carte.millisecondes)} ms"
    )
    ecart_carte = plus_grand_ecart(sur_le_processeur.reseau, sur_la_carte.reseau)
    print(f"Plus grand écart des poids finaux avec le processeur\u202f: {en_puissance_de_dix(ecart_carte)}")
else:
    print("Pas de carte graphique ici\u202f: PyTorch a tout calculé sur le processeur.")
    print("Dans Colab, pour avoir une carte\u202f: Exécution > Modifier le type d’exécution > GPU, puis relance le carnet depuis le début.")

## Exercice : un seul neurone caché suffit-il ?

Le réseau de ce carnet a 2 neurones cachés. Avec une couche cachée d’**un seul** neurone, le réseau apprend-il le OU exclusif ?

Écris ta réponse à la place de `None` : `True` si tu penses que oui, `False` si tu penses que non. Puis lance la cellule. Elle entraîne avec PyTorch un réseau 2-1-1, à partir de 5 graines, et compte ses bonnes réponses.

Indice : relis la fin de l’étape 1. Le neurone de sortie ne voit plus qu’un seul nombre, la sortie du neurone caché.

In [ ]:
# À toi : remplace None par True (il apprend le OU exclusif) ou par False (il ne l’apprend pas).
ma_reponse = None

# Ne change rien en dessous : cette partie entraîne un réseau 2-1-1 (un seul neurone caché)
# à partir de 5 graines, et garde le meilleur nombre de bonnes réponses sur les 4 points.
from torch import nn


def bonnes_reponses_avec_un_seul_neurone_cache(graine):
    torch.manual_seed(graine)  # les poids de départ de nn.Linear viennent de ce hasard
    petit = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid(), nn.Linear(1, 1), nn.Sigmoid()).double()
    x = torch.tensor(POINTS, dtype=torch.float64)
    y = torch.tensor([[reponse] for reponse in PORTES["ou_exclusif"]], dtype=torch.float64)
    optimiseur = torch.optim.SGD(petit.parameters(), lr=TAUX)
    for _ in range(NB_PAS):
        optimiseur.zero_grad()
        nn.BCELoss()(petit(x), y).backward()
        optimiseur.step()
    with torch.no_grad():
        return int((petit(x).round() == y).sum())


if ma_reponse is None:
    print("Pas encore. Remplace None par True ou par False, puis relance la cellule.")
else:
    meilleur = max(bonnes_reponses_avec_un_seul_neurone_cache(graine) for graine in range(1, 6))
    if ma_reponse is False:
        print(f"Défi réussi\u202f: avec un seul neurone caché, le réseau classe au mieux {meilleur} points sur 4, sur 5 graines.")
    else:
        print(f"Pas encore. Avec un seul neurone caché, le réseau classe au mieux {meilleur} points sur 4, sur 5 graines. Relis l’indice.")

<details>
<summary>Voir une correction (essaie d’abord !)</summary>

Non. Avec un seul neurone caché, tout passe par un seul nombre, h = sigmoïde(w1 × x1 + w2 × x2 + b). La sortie ne dépend que de h, et elle varie toujours dans le même sens quand h augmente : le réseau répond 1 d’un côté d’une valeur de h, et 0 de l’autre. Comme la sigmoïde monte toujours, cela revient à couper le plan des points par une seule ligne droite, comme le neurone à seuil de l’étape 1. Il classe donc 3 points sur 4 au mieux.

Avec deux neurones cachés, chacun trace sa propre ligne, et la sortie combine les deux : c’est ce qui permet de mettre (0, 1) et (1, 0) à part.

Remplace la première ligne de code de la cellule de l’exercice par :

```python
ma_reponse = False
```

</details>

## Dernière étape : les mêmes nombres que la page ?

La page du site ne lance pas Python. Son atelier refait le neurone à seuil en TypeScript, directement dans ton navigateur, et sa courbe montre les pertes d’un fichier de référence, `tests/attendus.json`, calculé une fois pour toutes par le Python. La cellule suivante recalcule ce fichier avec les fonctions du projet, puis le compare :

- au bit près, le neurone à seuil, les poids de départ et la liste des graines qui apprennent ;
- à 10⁻⁹ près, les 3 001 pertes et les 4 prédictions : elles passent par l’exponentielle et le logarithme, dont le dernier chiffre peut changer d’un ordinateur à l’autre.

Puis elle dit sur quoi ce carnet a tourné.

In [ ]:
import json
import math

from premier_reseau.attendus import CLES_LIBM, calculer_attendus, vers_json

fichier = json.loads((projet / "tests" / "attendus.json").read_text(encoding="utf-8"))
calcule = json.loads(vers_json(calculer_attendus()))  # recalculé ici, puis relu comme le fichier


def sans_libm(attendus):
    """Les attendus sans les pertes ni les prédictions."""
    entrainement = {cle: valeur for cle, valeur in attendus["entrainement"].items() if cle not in CLES_LIBM}
    return {**attendus, "entrainement": entrainement}


# Au bit près, par le texte JSON : « == » prendrait -0.0 pour 0.0, et 6 pour 6.0.
exact = vers_json(sans_libm(calcule)) == vers_json(sans_libm(fichier))
# À 10⁻⁹ près, nombre par nombre, pour ce qui passe par exp et log.
proches = all(
    len(calcule["entrainement"][cle]) == len(fichier["entrainement"][cle])
    and all(
        math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-12)
        for a, b in zip(calcule["entrainement"][cle], fichier["entrainement"][cle])
    )
    for cle in CLES_LIBM
)
reussites = sum(graine["reussit"] for graine in calcule["graines"])
print(f"Graines qui apprennent le OU exclusif\u202f: {reussites} sur {len(calcule['graines'])}.")
print(f"{en_francais(len(calcule['entrainement']['pertes']))} pertes comparées à 10⁻⁹ près, et le reste au bit près.")
print(f"Même résultat que la page du site\u202f: {'oui' if exact and proches else 'non'}")

if ou.type == "cuda":
    print(f"Ce carnet a tourné sur\u202f: la carte graphique, {torch.cuda.get_device_name(0)}.")
else:
    print("Ce carnet a tourné sur\u202f: le processeur.")  # \u202f est l’espace fine insécable

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/0.5-reseau sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/0.5-reseau).
- La page du chapitre, avec l’atelier du neurone : [chapitre 0.5](https://chipat-neko.github.io/forge-ia/partie-0/0-5/).
- Sur ton ordinateur, dans le dossier `projets/0.5-reseau` : `uv run --extra cpu python -m premier_reseau entrainer` refait ce carnet d’un coup, du neurone à seuil à PyTorch. Avec une carte NVIDIA : `uv run --extra gpu python -m premier_reseau entrainer --appareil cuda`.
- La suite de La Forge IA entraîne des réseaux bien plus grands, avec les mêmes gestes : propager, mesurer l’erreur, rétropropager, faire un pas.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.